In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
import numpy as np
import math
import seaborn as sns
import pyarrow as pa
import pyarrow.parquet as pq
import itertools
import evi_functions as evi_func
import matplotlib.dates as mdates
import warnings
from pathlib import Path
from datetime import datetime

import descri_function as des_fun

import functions_mem as fm
import functions  # auxiliary warning-cleaning utilities

warnings.filterwarnings("ignore")


# Ler dado com base sintética

In [17]:
df = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/sintetic_outbreak_20260325.parquet')

In [18]:
dta = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/cities_valid_for_MEM_26_03_2026.parquet')

# Select cities for the manuscript analysis (valid MEM)
#lst = list(set(df.co_ibge.unique()) - set(dta.co_ibge.unique()))
lst = dta.co_ibge.unique()
df = df[df.co_ibge.isin(lst)]

df =  df[(df.year_week >= '2022-42') &(df.year_week <= '2025-32')]

# Ler dado do MEM

In [19]:
DATA_PATH = "/opt/storage/shared/aesop/aesop_shared/ensamble_modelling"

MEM_OUTPUT_FILE = max(Path(DATA_PATH).glob("mem_output_*parquet"), key=lambda x: x.stat().st_mtime)

# MEM thresholds and parameters per municipality
df_mem = pd.read_parquet(Path(DATA_PATH) / MEM_OUTPUT_FILE)

# Criar coluna de surtos nas séries artificiais para cada replica de cada municipio, baseado no MEM

In [22]:
def compute_mem_for_replicates(df_raw, df_mem, functions):
    results = []

    # detect replicate signal columns (NOT surge columns)
    rep_cols = [c for c in df_raw.columns if c.startswith('replicate_')]

    for co_ibge in df_mem.co_ibge.unique():
        print(f"Processing municipality {co_ibge}...")

        try:
            set_muni = df_raw[df_raw.co_ibge == co_ibge].copy()

            # MEM threshold
            epidemic_threshold = df_mem.loc[
                df_mem.co_ibge == co_ibge, "epidemic_threshold"
            ].max()

            threshold_base = epidemic_threshold

            for col in rep_cols:

                # extract replicate id (e.g. "replicate_3" → "3")
                rep_id = col.split('_')[1]

                # -------------------------------
                # STEP 1: MEM binary signal
                # -------------------------------
                mem_col = f"mem_surge_01_replicate_{rep_id}"

                set_muni[mem_col] = pd.cut(
                    set_muni[col],  #  THIS is the key fix
                    bins=[-np.inf, threshold_base, np.inf],
                    labels=[0, 1],
                    include_lowest=True,
                ).astype(int)

                # -------------------------------
                # STEP 2: clean signal
                # -------------------------------
                cleaned = functions.clean_warning_column(
                    set_muni,
                    group_col="co_ibge",
                    time_col="year_week",
                    warning_col=mem_col,
                )

                # -------------------------------
                # STEP 3: store outputs
                # -------------------------------
                set_muni[f"{mem_col}_without_isolated"] = cleaned["cleaned_warning"]
                set_muni[f"{mem_col}_correct_with_consec"] = cleaned["event"]
                set_muni[f"warning_final_{mem_col}"] = cleaned["warning_final"]

            results.append(set_muni)

        except Exception as e:
            print(f"Skipping {co_ibge} due to error: {e}")
            continue

    return pd.concat(results, ignore_index=True)

In [23]:
result = compute_mem_for_replicates(df, df_mem, functions)

Processing municipality 110001...
Processing municipality 110002...
Processing municipality 110005...
Processing municipality 110007...
Processing municipality 110008...
Processing municipality 110009...
Processing municipality 110010...
Processing municipality 110011...
Processing municipality 110013...
Processing municipality 110014...
Processing municipality 110015...
Processing municipality 110018...
Processing municipality 110020...
Processing municipality 110025...
Processing municipality 110026...
Processing municipality 110028...
Processing municipality 110029...
Processing municipality 110030...
Processing municipality 110032...
Processing municipality 110033...
Processing municipality 110034...
Processing municipality 110037...
Processing municipality 110040...
Processing municipality 110045...
Processing municipality 110050...
Processing municipality 110060...
Processing municipality 110080...
Processing municipality 110090...
Processing municipality 110092...
Processing mun

In [26]:
result.columns.to_list()

['replicate_0',
 'replicate_1',
 'replicate_2',
 'replicate_3',
 'replicate_4',
 'replicate_5',
 'replicate_6',
 'replicate_7',
 'replicate_8',
 'replicate_9',
 'replicate_10',
 'replicate_11',
 'replicate_12',
 'replicate_13',
 'replicate_14',
 'replicate_15',
 'replicate_16',
 'replicate_17',
 'replicate_18',
 'replicate_19',
 'replicate_20',
 'replicate_21',
 'replicate_22',
 'replicate_23',
 'replicate_24',
 'replicate_25',
 'replicate_26',
 'replicate_27',
 'replicate_28',
 'replicate_29',
 'replicate_30',
 'replicate_31',
 'co_ibge',
 'year_week',
 'atend_ivas',
 'mem_surge_01_correct_with_consec',
 'warning_final_mem_surge_01',
 'mem_surge_01_replicate_0',
 'mem_surge_01_replicate_0_without_isolated',
 'mem_surge_01_replicate_0_correct_with_consec',
 'warning_final_mem_surge_01_replicate_0',
 'mem_surge_01_replicate_1',
 'mem_surge_01_replicate_1_without_isolated',
 'mem_surge_01_replicate_1_correct_with_consec',
 'warning_final_mem_surge_01_replicate_1',
 'mem_surge_01_replicat

In [27]:
#result.to_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/sintetic_with_MEM_surge_col_29_04_2026.parquet')

In [20]:
# Index MEM for performance
df_mem_indexed = df_mem.set_index("co_ibge")

# ============================================================
# Apply MEM-based surge definitions
# ============================================================

results_mem = []

for co_ibge in df_mem.co_ibge.unique():
    print(f"Processing municipality {co_ibge}...")

    try:
        set_muni = df_raw[df_raw.co_ibge == co_ibge].copy()

        # Retrieve MEM thresholds
        epidemic_threshold = df_mem.loc[df_mem.co_ibge == co_ibge, "epidemic_threshold"].max()

        # ----------------------------------------------------
        # Define surge thresholds
        # ----------------------------------------------------
        threshold_base = epidemic_threshold 

        # Binary MEM surge (0/1)
        set_muni["mem_surge_01"] = pd.cut( ## mem_surge_01_replicate_*
                set_muni['replicate_*'],
                bins=[-np.inf, threshold_base, np.inf],
                labels=[0, 1],
                include_lowest=True,
            ).astype(int)

        results_mem.append(set_muni)

    except Exception as e:
        print(f"⚠️ Skipping {co_ibge} due to error: {e}")
        continue


df_mem_surge = pd.concat(results_mem, ignore_index=True)


# ============================================================
# Clean isolated warnings and enforce temporal consistency
# ============================================================

df_mem_cleaned = functions.clean_warning_column(
    df_mem_surge,
    group_col="co_ibge",
    time_col="year_week",
    warning_col="mem_surge_01",
)

df_mem_cleaned = df_mem_cleaned.rename(
    columns={
        "cleaned_warning": "mem_surge_01_without_isolated", # needs to rename accordinf to replicate
        "event": "mem_surge_01_correct_with_consec",   # needs to rename accordinf to replicate
        "warning_final": "warning_final_mem_surge_01",  # needs to rename accordinf to replicate
    }
)

,replicate_0,replicate_1,replicate_2,replicate_3,replicate_4,replicate_5,replicate_6,replicate_7,replicate_8,replicate_9,...,replicate_27,replicate_28,replicate_29,replicate_30,replicate_31,co_ibge,year_week,atend_ivas,mem_surge_01_correct_with_consec,warning_final_mem_surge_01
302,31,31,31,31,31,31,31,31,31,31,...,31,31,31,31,31,110001,2022-42,31,0,0
303,16,16,16,16,16,16,16,16,16,16,...,16,16,16,16,16,110001,2022-43,16,0,0
304,20,20,20,20,20,20,20,20,20,20,...,20,20,20,20,20,110001,2022-44,20,0,0
305,32,32,32,32,32,32,32,32,32,32,...,32,32,32,32,32,110001,2022-45,32,0,0
306,47,47,47,47,47,47,47,47,47,47,...,47,47,47,47,47,110001,2022-46,47,0,0


In [ ]:

# ============================================================
# Alternative surge definition (municipalities without MEM)
# ============================================================

missing_mem = set(df_raw.co_ibge.unique()) - set(df_mem_cleaned.co_ibge.unique())

fallback_results = []

for co_ibge in missing_mem:

    set_muni = df_raw[df_raw.co_ibge == co_ibge].copy()

    # Simple statistical baseline
    mean_val = set_muni.atend_ivas.mean()
    median_val = set_muni.atend_ivas.median()
    std_val = set_muni.atend_ivas.std()

    set_muni["surge_ivas_alternative"] = (
        set_muni["atend_ivas"] > median_val + 2 * std_val
    ).astype(int)

    fallback_results.append(set_muni)


df_fallback = pd.concat(fallback_results, ignore_index=True)

df_fallback_cleaned = functions.clean_warning_column(
    df_fallback,
    group_col="co_ibge",
    time_col="year_week",
    warning_col="surge_ivas_alternative",
)

df_fallback_cleaned = df_fallback_cleaned.rename(
    columns={
        "cleaned_warning": "mem_surge_01_without_isolated",
        "event": "mem_surge_01_correct_with_consec",
        "warning_final": "warning_final_mem_surge_01",
    }
)


# ============================================================
# Merge MEM-based and fallback results
# ============================================================

final_mem = df_mem_cleaned[
    [
        "co_uf",
        "nm_uf",
        "nm_municipio",
        "co_ibge",
        "epiyear",
        "epiweek",
        "year_week",
        "atend_totais",
        "atend_ivas",
        'mem_surge_01_correct_with_consec',
        "warning_final_mem_surge_01",
    ]
]

final_fallback = df_fallback_cleaned[
    [
        "co_uf",
        "nm_uf",
        "nm_municipio",
        "co_ibge",
        "epiyear",
        "epiweek",
        "year_week",
        "atend_totais",
        "atend_ivas",
        'mem_surge_01_correct_with_consec',
        "warning_final_mem_surge_01",
    ]
]

final = pd.concat([final_mem, final_fallback], ignore_index=True)







In [ ]:
# ============================================================
# Save output
# ============================================================

out_file = (
    Path(DATA_PATH)
    / f"{OUTPUT_PREFIX}_{datetime.now():%d_%m_%Y}.parquet"
)

final.to_parquet(out_file)

print(f"Saved final MEM warning dataset to: {out_file}")